In [1]:
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

data_dir = Path('../data')
df = pd.read_csv(data_dir / 'raw/bank-additional-full.csv', header=0, sep=';')
print(df.shape)
print(df['y'].value_counts())
print(df.info())

(41188, 21)
y
no     36548
yes     4640
Name: count, dtype: int64
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41188 entries, 0 to 41187
Data columns (total 21 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   age             41188 non-null  int64  
 1   job             41188 non-null  object 
 2   marital         41188 non-null  object 
 3   education       41188 non-null  object 
 4   default         41188 non-null  object 
 5   housing         41188 non-null  object 
 6   loan            41188 non-null  object 
 7   contact         41188 non-null  object 
 8   month           41188 non-null  object 
 9   day_of_week     41188 non-null  object 
 10  duration        41188 non-null  int64  
 11  campaign        41188 non-null  int64  
 12  pdays           41188 non-null  int64  
 13  previous        41188 non-null  int64  
 14  poutcome        41188 non-null  object 
 15  emp.var.rate    41188 non-null  float64
 16  cons.price

In [2]:
# Missing values
print(df.isnull().sum())


age               0
job               0
marital           0
education         0
default           0
housing           0
loan              0
contact           0
month             0
day_of_week       0
duration          0
campaign          0
pdays             0
previous          0
poutcome          0
emp.var.rate      0
cons.price.idx    0
cons.conf.idx     0
euribor3m         0
nr.employed       0
y                 0
dtype: int64


In [3]:

# Count "unknown" values in categorical features
categorical_cols = df.select_dtypes(
    include=["object"]
).columns

for col in categorical_cols:
    unknown_count = (df[col] == "unknown").sum()
    if unknown_count > 0:
        print(f"{col}: {unknown_count}")

job: 330
marital: 80
education: 1731
default: 8597
housing: 990
loan: 990


In [4]:
X = df.drop(columns=["y", "duration"], axis=1)
y = df["y"].map({
    "no": 0,
    "yes": 1
})

print("Features:", X.shape)
print("Target:", y.shape)
print("Target distribution:")
print(y.value_counts(normalize=True))

Features: (41188, 19)
Target: (41188,)
Target distribution:
y
0    0.887346
1    0.112654
Name: proportion, dtype: float64


In [5]:
numeric_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object"]
).columns.tolist()

print("Numeric:", numeric_features)
print("Categorical:", categorical_features)

Numeric: ['age', 'campaign', 'pdays', 'previous', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed']
Categorical: ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'day_of_week', 'poutcome']


In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

X_valid, X_test, y_valid, y_test = train_test_split(
    X_test, y_test, test_size=0.5, random_state=42, stratify=y_test
)

print("Train:", X_train.shape)
print("Validation:", X_valid.shape)
print("Test:", X_test.shape)


for name, labels in [("Train", y_train), ("Validation", y_valid), ("Test", y_test)]:
    print(name, labels.mean())

Train: (28831, 19)
Validation: (6178, 19)
Test: (6179, 19)
Train 0.11265651555617218
Validation 0.11265781806409841
Test 0.11263958569347791


In [7]:

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])

model.fit(X_train, y_train)

y_pred = model.predict(X_valid)

y_prob = model.predict_proba(X_valid)[:, 1]

print("Predictions:", y_pred[:10])
print("Probabilities:", y_prob[:10])


Predictions: [0 0 0 0 0 0 0 0 0 0]
Probabilities: [0.04624539 0.06942053 0.04293895 0.04719964 0.10957867 0.13334475
 0.05013543 0.04095024 0.05983558 0.04890982]


In [8]:

from sklearn.metrics import (classification_report, roc_auc_score, average_precision_score, confusion_matrix)

print(classification_report(y_valid, y_pred, zero_division=0))

print("ROC-AUC:", roc_auc_score(y_valid, y_prob))
print("PR-AUC:", average_precision_score(y_valid, y_prob))

print("Confusion matrix:")
print(confusion_matrix(y_valid, y_pred))


              precision    recall  f1-score   support

           0       0.91      0.99      0.95      5482
           1       0.69      0.22      0.33       696

    accuracy                           0.90      6178
   macro avg       0.80      0.60      0.64      6178
weighted avg       0.88      0.90      0.88      6178

ROC-AUC: 0.8021599948839883
PR-AUC: 0.4700697725957582
Confusion matrix:
[[5414   68]
 [ 543  153]]


In [11]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier

models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        min_samples_leaf=5,
        random_state=42,
        n_jobs=-1
    ),

    "XGBoost": XGBClassifier(
        n_estimators=200,
        max_depth=4,
        learning_rate=0.05,
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    )
}

In [12]:
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)
import pandas as pd

results = []
trained_models = {}

for name, classifier in models.items():

    pipeline = Pipeline([
        ("preprocessor", clone(preprocessor)),
        ("classifier", classifier)
    ])

    # Train
    pipeline.fit(X_train, y_train)

    # Predict on validation data
    y_pred = pipeline.predict(X_valid)
    y_prob = pipeline.predict_proba(X_valid)[:, 1]

    # Evaluate
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_valid, y_pred),
        "F1": f1_score(y_valid, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_valid, y_prob),
        "PR-AUC": average_precision_score(y_valid, y_prob)
    })

    trained_models[name] = pipeline

results_df = pd.DataFrame(results)

print(
    results_df.sort_values(
        by="PR-AUC",
        ascending=False
    ).round(4)
)

                 Model  Accuracy      F1  ROC-AUC  PR-AUC
2              XGBoost    0.9008  0.3541   0.8131  0.4948
1        Random Forest    0.9026  0.3716   0.8038  0.4896
0  Logistic Regression    0.9011  0.3337   0.8022  0.4701


In [13]:
best_name = results_df.sort_values(
    by="PR-AUC",
    ascending=False
).iloc[0]["Model"]

best_model = trained_models[best_name]

print("Best model:", best_name)

Best model: XGBoost


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.1/119.1 kB 1.5 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 2.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 397.6/397.6 kB 2.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.7/216.7 kB 2.0 MB/s eta 0:00:00a 0:00:01
